<a href="https://colab.research.google.com/github/ArtSharan/SDC_CODES/blob/main/CAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Install Required Libraries (if needed)
!pip install -q tensorflow_addons

# Step 2: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
import tensorflow_addons as tfa

# Step 3: Load MNIST Data
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize and reshape
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# Step 4: Squash Function for Capsules
def squash(vectors, axis=-1):
    s_squared_norm = tf.reduce_sum(tf.square(vectors), axis, keepdims=True)
    scale = s_squared_norm / (1 + s_squared_norm) / tf.sqrt(s_squared_norm + tf.keras.backend.epsilon())
    return scale * vectors

# Step 5: Primary Capsule Layer
class PrimaryCapsule(layers.Layer):
    def __init__(self, num_capsules, dim_capsule, **kwargs):
        super(PrimaryCapsule, self).__init__(**kwargs)
        self.conv = layers.Conv2D(filters=num_capsules * dim_capsule, kernel_size=9, strides=2, padding='valid')
        self.num_capsules = num_capsules
        self.dim_capsule = dim_capsule

    def call(self, inputs):
        output = self.conv(inputs)
        output = tf.reshape(output, (-1, self.num_capsules * 6 * 6, self.dim_capsule))
        return squash(output)

# Step 6: Digit Capsule Layer
class DigitCapsule(layers.Layer):
    def __init__(self, num_capsules, dim_capsule, routings=3, **kwargs):
        super(DigitCapsule, self).__init__(**kwargs)
        self.num_capsules = num_capsules
        self.dim_capsule = dim_capsule
        self.routings = routings

    def build(self, input_shape):
        self.W = self.add_weight(shape=[1, input_shape[1], self.num_capsules, self.dim_capsule, input_shape[2]],
                                 initializer='glorot_uniform', trainable=True)

    def call(self, inputs):
        inputs_expand = tf.expand_dims(inputs, 2)
        inputs_expand = tf.expand_dims(inputs_expand, 4)
        inputs_tiled = tf.tile(inputs_expand, [1, 1, self.num_capsules, 1, 1])
        u_hat = tf.matmul(self.W, inputs_tiled)
        u_hat_stopped = tf.stop_gradient(u_hat)

        b = tf.zeros(shape=[tf.shape(inputs)[0], inputs.shape[1], self.num_capsules, 1])

        for i in range(self.routings):
            c = tf.nn.softmax(b, axis=2)
            if i == self.routings - 1:
                s = tf.reduce_sum(c * u_hat, axis=1, keepdims=True)
                v = squash(s, axis=-2)
            else:
                s = tf.reduce_sum(c * u_hat_stopped, axis=1, keepdims=True)
                v = squash(s, axis=-2)
                b += tf.reduce_sum(u_hat_stopped * v, axis=-1, keepdims=True)

        return tf.squeeze(v, axis=1)

# Step 7: Create Capsule Network Model
def CapsNet():
    inputs = layers.Input(shape=(28, 28, 1))
    conv1 = layers.Conv2D(256, 9, activation='relu')(inputs)
    primary_caps = PrimaryCapsule(32, 8)(conv1)
    digit_caps = DigitCapsule(10, 16)(primary_caps)
    out_caps = tf.norm(digit_caps, axis=-1)

    model = models.Model(inputs, out_caps)
    return model

# Step 8: Compile & Train
model = CapsNet()
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=5, batch_size=128)

# Step 9: Evaluate
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"\nTest Accuracy: {test_acc:.4f}")